# Project out all exit points for each entry point

* Entry points are 10 AM to 3 PM of options w/ 2+ TV 5 < IV
* Holding duration is 2 hours max


Create entry point view.  This has not selection creiteria imdeded at this point.

In [1]:
! pip install sqlalchemy pandas mysqlclient  # or pymysql

You should consider upgrading via the '/Users/Muthu/Development/OptionList11/bin/bin/python -m pip install --upgrade pip' command.


In [2]:
import sqlalchemy as sa
import pandas as pd
from sqlalchemy.sql import text

In [3]:
# Replace with your own database credentials
username = 'rk_admin'
password = 'rk2admin!'
host = 'localhost'
database = 'ol7'

# For mysqlclient or pymysql
engine = sa.create_engine(f'mysql+pymysql://{username}:{password}@{host}/{database}')
conn = engine.connect()


OperationalError: (pymysql.err.OperationalError) (2003, "Can't connect to MySQL server on 'localhost' ([Errno 8] nodename nor servname provided, or not known)")
(Background on this error at: https://sqlalche.me/e/20/e3q8)

In [4]:
# SQL Query
query = "SELECT * FROM stock_list"

# Use pandas to read the query result into a DataFrame
df = pd.read_sql(query, engine)

df.head()

OperationalError: (pymysql.err.OperationalError) (2003, "Can't connect to MySQL server on 'localhost' ([Errno 8] nodename nor servname provided, or not known)")
(Background on this error at: https://sqlalche.me/e/20/e3q8)

In [5]:
with engine.connect() as connection:
    connection.execute(text("drop view if exists open_options"))

OperationalError: (pymysql.err.OperationalError) (2003, "Can't connect to MySQL server on 'localhost' ([Errno 8] nodename nor servname provided, or not known)")
(Background on this error at: https://sqlalche.me/e/20/e3q8)

In [14]:
with engine.connect() as connection:
    connection.execute(text('''create view open_options(
    op_date, op_sq_id, op_oq_id,
    op_sq_trade_average, op_sq_bid_average, op_sq_ask_average, op_sq_trade_average_delta_30, op_sq_barcount_sum_30,
    op_oq_trade_average, op_oq_bid_average, op_oq_ask_average, op_oq_trade_average_delta_30, op_oq_barcount_sum_30,
    op_iv, op_tv, op_dur,
    ol_con_id, ol_expiry, ol_strike, ol_local_symbol) as
select sq.quote_date op_date, sq.id op_sq_id,  oq.id op_oq_id,
       sq.trade_average op_sq_trade_average, sq.bid_avg , sq.ask_avg, format(sq.trade_average_delta_30,3) op_sq_trade_average_delta_30,
       sq.barcount_sum_30 op_sq_barcount_sum_30,

       oq.trade_average op_oq_trade_average, oq.bid_avg , oq.ask_avg, format(oq.trade_average_delta_30,3) op_oq_trade_average_delta_30,
       oq.barcount_sum_30 op_oq_barcount_sum_30,

       oq.iv op_iv, oq.open_tv op_tv, get_min_diff(sq.quote_date, ol.expiry) op_dur,
       ol.con_id ol_con_id,  ol.expiry ol_expiry, ol.strike ol_strike,  ol.local_symbol ol_local_symbol
from  option_list ol, option_quote oq, stock_quote sq
where sq.quote_date = oq.quote_date
and oq.con_id = ol.con_id
and ol.option_type = 'C'

order by oq.con_id, sq.quote_date'''))